<h1><center>SpaGCN Tutorial</center></h1>


<center>Author: Jian Hu,*, Xiangjie Li, Kyle Coleman, Amelia Schroeder, Nan Ma, David J. Irwin, Edward B. Lee, Russell T. Shinohara, Mingyao Li*

### Outline
1. Installation
2. Import modules
3. Read in data
4. Integrate gene expression and histology into a Graph
5. Spatial domain detection using SpaGCN
6. Identify SVGs
7. Identify Meta Gene
8. Multiple tissue sections analysis

### 1. Installation
To install SpaGCN package you must make sure that your python version is over 3.5.=. If you don’t know the version of python you can check it by:

In [1]:
import platform
platform.python_version()

'3.8.8'

Note: Because SpaGCN pends on pytorch, you should make sure torch is correctly installed.
<br>
Now you can install the current release of SpaGCN by the following three ways:
#### 1.1 PyPI: Directly install the package from PyPI.

In [ ]:
pip3 install SpaGCN
#Note: you need to make sure that the pip is for python3，or we should install SpaGCN by
python3 -m pip install SpaGCN
pip3 install SpaGCN
#If you do not have permission (when you get a permission denied error), you should install SpaGCN by
pip3 install --user SpaGCN

#### 1.2 Github
Download the package from Github and install it locally:

In [2]:
git clone https://github.com/jianhuupenn/SpaGCN
cd SpaGCN/SpaGCN_package/
python3 setup.py install --user

#### 1.3 Anaconda
If you do not have Python3.5 or Python3.6 installed, consider installing Anaconda (see Installing Anaconda). After installing Anaconda, you can create a new environment, for example, SpaGCN (you can change to any name you like).

In [ ]:
#create an environment called SpaGCN
conda create -n SpaGCN python=3.7.9
#activate your environment 
conda activate SpaGCN
git clone https://github.com/jianhuupenn/SpaGCN
cd SpaGCN/SpaGCN_package/
python3 setup.py build
python3 setup.py install
conda deactivate

### 2. Import python modules

In [1]:
import os,csv,re
import pandas as pd
import numpy as np
import scanpy as sc
import math
import SpaGCN as spg
from scipy.sparse import issparse
import random, torch
import warnings
warnings.filterwarnings("ignore")
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import SpaGCN as spg
#In order to read in image data, we need to install some package. Here we recommend package "opencv"
#inatll opencv in python
#!pip3 install opencv-python
import cv2

/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
spg.__version__

'1.2.7'

In [3]:
# Paths and variables
person = "CRC_P1"
bin_size = "002um"
os.makedirs(f"../data/{person}", exist_ok=True)
local_data_path = f"../data/{person}"

genomosaic_input_path = f"../../../data/{person}/input"
genomosaic_output_path = f"../../../data/{person}/output/binned_outputs/square_{bin_size}/spatial"
cell_seg_path = f"../../../Cell_Segmentation/output/{person}/{bin_size}"

imagepath = os.path.join(genomosaic_input_path, "Visium_HD_Human_Colon_Cancer_P1_tissue_image.btf")
os.makedirs(f"../results/{person}", exist_ok=True)
results_path = f"../results/{person}"

## 3. Read in data
The current version of SpaGCN requres three input data: 
<br>
1. The gene expression matrix(n by k): expression_matrix.h5;
<br>
2. Spatial coordinateds of samplespositions.txt;
<br>
3. Histology image(optional): histology.tif, can be tif or png or jepg.
<br>
The gene expreesion data can be stored as an AnnData object. AnnData stores a data matrix .X together with annotations of observations .obs, variables .var and unstructured annotations .uns. 

### 3.1 Calculate centroids for nuclei
For that, we read in the parquet file with the geometries for each nucleus.
The current approach is to just use .centroid to determine the centroid and store it as *pixel* point coordinates.  
NOTE: the nucleus coordinates ONLY exactly match `count_area_filtered_adata.h5ad`. `nuclei_grouped_UMI_count.h5ad` has MORE nuclei, which polygons can be found one to two folders above. 

In [4]:
# Note: cell added by me 
# Loads both the anndata file and the parquet file containing the nuclei polygons.
# Then: calculates nuclei centroids as pixel coordinates and store in new dataframe tissue_positions
import pandas as pd
import geopandas as gpd
import gc


nuclei_shapes = gpd.read_parquet(os.path.join(cell_seg_path, "custom_binning/nuclei_geometry.parquet"))
adata=sc.read(os.path.join(cell_seg_path, "custom_binning/count_area_filtered_adata.h5ad"))

if adata.obs["id"].is_unique:
    adata.obs = adata.obs.set_index("id")
else:
    print("WARNING: the nucleus IDs are not unique. Please do not continue until fixed.")

# calc centroids from nuclei polygons
nuclei_shapes["centroid"] = nuclei_shapes.geometry.centroid

# make nuclei ID column index column
if nuclei_shapes["id"].is_unique:
    nuclei_shapes = nuclei_shapes.set_index("id")
else:
    print("WARNING: Nuclei IDs not unique.")

nuclei_positions = nuclei_shapes.index.to_frame(name="id").copy()
nuclei_positions = nuclei_positions.drop(columns="id")

nuclei_positions["x_pixel"] = [pt.x for pt in nuclei_shapes["centroid"]]
nuclei_positions["y_pixel"] = [pt.y for pt in nuclei_shapes["centroid"]]

assert nuclei_positions[(nuclei_positions["x_pixel"] < 0) | (nuclei_positions["y_pixel"] < 0)].shape[0] == 0, "Nuclei must have positive pixel coordinates."

del nuclei_shapes
gc.collect()



11

### 3.2 Calculate Array Positions of Nucleus Centroids based on Pixels

We now also need the array coordinates. 
Since I could not find any direct conversion, I will do it using the original tissue positions.parquet file, as it seems that each 2x2 bin is one array cell.
I then vectorise it and enter the calculated array coordinates into the dataframe and save it as a temporary csv. 

**VERY IMPORTANT NOTE**  
The array grid is layered in a standarad fashion, such that (0,0) correpsonds to the point at the *bottom left* of the tissue frame, like in a standard cooridnate system.  
HOWEVER, the pixel coordinate system starts from the *Top Left*. 
The x-axis hence increases in the same direction, but for the array the y-axis increases upwards while it increases downwards for the pixel coordinate system.

Hence, for pixel to array coordinate conversion it holds:  
Array[x_min, y_min] = Pixel[x_min, y_max] and  Pixel[x_min, y_min]  --> Array[x_min, y_max]  
Array[x_max, y_max] = Pixel[x_max, y_min] and  Pixel[x_max, y_max]  --> Array[x_max, y_min]   
row --> y and col --> x

In [5]:
# Note: cell added by me
# Gets original array and pixel positions per bin (bin = 1 array cell) and bin width
# calculates array width and length in cells
import json

tissue_positions = pd.read_parquet(os.path.join(genomosaic_output_path, "tissue_positions.parquet"))
scale_path = os.path.join(genomosaic_output_path, "scalefactors_json.json")
with open(scale_path, "r") as f:
    scales = json.load(f)

array_x_min = tissue_positions["array_col"].min()
array_y_min = tissue_positions["array_row"].min()
array_x_max = tissue_positions["array_col"].max()
array_y_max = tissue_positions["array_row"].max()

# +1, as rows and columns are counted from 0 on
n_rows = array_y_max - array_y_min + 1
n_cols = array_x_max - array_x_min + 1

print("Min: ", array_x_min, array_y_min)
print("Max: ", array_x_max, array_y_max)

tissue_pos_array = tissue_positions[((tissue_positions["array_col"] == array_x_min) | 
                                     (tissue_positions["array_col"] == array_x_max)) &
                                    ((tissue_positions["array_row"] == array_y_min) |
                                     (tissue_positions["array_row"] == array_y_max))]

Min:  0 0
Max:  3349 3349


In [6]:
# return pixel column or row coordinate based on array values
def get_pxl_value(tissue_pos, col_val, row_val):
    col_val_pxl = tissue_pos.loc[(tissue_pos["array_row"] == row_val) & (tissue_pos["array_col"] == col_val), "pxl_col_in_fullres"].iloc[0]
    row_val_pxl = tissue_pos.loc[(tissue_pos["array_row"] == row_val) & (tissue_pos["array_col"] == col_val), "pxl_row_in_fullres"].iloc[0]
    return col_val_pxl, row_val_pxl


px_top_left, py_top_left = get_pxl_value(tissue_pos_array, array_x_min, array_y_max)
px_top_right, py_top_right = get_pxl_value(tissue_pos_array, array_x_max, array_y_max)
px_bottom_left, py_bottom_left = get_pxl_value(tissue_pos_array, array_x_min, array_y_min)
px_bottom_right, py_bottom_right = get_pxl_value(tissue_pos_array, array_x_max, array_y_min)

pixel_width = px_top_right - px_top_left
pixel_height = py_bottom_left - py_top_left  # vertical distance downward


Reminder: cooordinates are applied in (y, x)-fashion  

Pxl      | Array  
< - - - - - - - - - - - - -   
0, 0          | max, 0  
0,max   | max, max  
max,0   | 0, 0  
max,max | 0, max

In [7]:
array_pts = np.array([
    [array_x_min, array_y_min],  # bottom-left
    [array_x_max, array_y_min],  # bottom-right
    [array_x_min, array_y_max],  # top-left
    [array_x_max, array_y_max],  # top-right
])

pixel_pts = np.array([
    [px_bottom_left,  py_bottom_left],   # bottom-left
    [px_bottom_right, py_bottom_right],  # bottom-right
    [px_top_left,     py_top_left],      # top-left
    [px_top_right,    py_top_right],     # top-right
])

# === 2. Solve for affine transform coefficients ===
# We model: [array_x, array_y] = [x_pixel, y_pixel, 1] @ params
A = np.hstack([pixel_pts, np.ones((4, 1))])  # shape (4, 3)
B = array_pts                                # shape (4, 2)

params, _, _, _ = np.linalg.lstsq(A, B, rcond=None)  # shape (3, 2)
# params.T gives a 2×3 matrix you can use directly

# === 3. Apply affine transform to all pixel points ===
px = nuclei_positions['x_pixel'].to_numpy()
py = nuclei_positions['y_pixel'].to_numpy()
ones = np.ones_like(px)

pixel_matrix = np.vstack([px, py, ones]).T  # shape (N, 3)
array_coords = pixel_matrix @ params         # shape (N, 2)

array_x = np.floor(array_coords[:, 0]).astype(int)
array_y = np.floor(array_coords[:, 1]).astype(int)

# === 4. Clamp to grid boundaries and store ===
array_x = np.clip(array_x, array_x_min, array_x_max)
array_y = np.clip(array_y, array_y_min, array_y_max)

nuclei_positions['array_col'] = array_x
nuclei_positions['array_row'] = array_y

assert nuclei_positions[(nuclei_positions["array_row"] < 0) | (nuclei_positions["array_col"] < 0)].shape[0] == 0, "Nuclei must have positive array coordinates."


In [8]:
# Ensure index uniqueness
assert adata.obs.index.is_unique, "adata.obs index is not unique!"
assert nuclei_positions.index.is_unique, "df index is not unique!"

# Align and merge
adata.obs = adata.obs.join(nuclei_positions, how="left")
# For some reason: add column "genenames" ??
adata.var_names=[i.upper() for i in list(adata.var_names)]  # force uppercase
adata.var["genename"]=adata.var.index.astype("str")

In [ ]:
# Save the data from the previous cells
# nuclei_positions.to_csv(f"../data/{person}/nuclei_positions.csv")
# adata.write_h5ad(f"../data/{person}/Nuclei_centroids.h5ad")

### 3.3 Load Data (again)

Load adata agin if you have done the previous steps at another point in time.  
Load the image, but crop to only the section needed, as the full image is 12GB+ in size.

In [4]:
import math
import pyvips
import numpy as np
import cv2
import gc
import os

nuclei_centroids_path = os.path.join(local_data_path, "nuclei_centroids.h5ad")

#Read in gene expression and spatial location
adata=sc.read(nuclei_centroids_path)
#tissuepositions to get only needed pixel space from tif to reduce size
tissue_positions = pd.read_parquet(f"../../../data/{person}/output/binned_outputs/square_{bin_size}/spatial/tissue_positions.parquet")
xmin = max(0, math.floor(tissue_positions["pxl_col_in_fullres"].min()))
xmax = max(0, math.ceil(tissue_positions["pxl_col_in_fullres"].max()))
ymin = max(0, math.floor(tissue_positions["pxl_row_in_fullres"].min()))
ymax = max(0, math.ceil(tissue_positions["pxl_row_in_fullres"].max()))

print("New image coordinates: ", xmin, xmax, ymin, ymax)
assert xmin >= 0 and xmax >= 0 and ymin >= 0 and ymax >0 and (xmin != xmax | ymin != ymax), "Image pixel coordinates can not be smaller than 0 and total area must be greater than 0."


# Open + crop image
image = pyvips.Image.new_from_file(imagepath, access="sequential")
cropped = image.crop(xmin, ymin, xmax - xmin, ymax - ymin)
del image

dtype_map = {
    'uchar': np.uint8,
    'char': np.int8,
    'ushort': np.uint16,
    'short': np.int16,
    'uint': np.uint32,
    'int': np.int32,
    'float': np.float32,
    'double': np.float64,
}
# Avoid loss if image has large number of channels
dtype = dtype_map[cropped.format]

# Convert to NumPy array
numpy_array = np.ndarray(
    buffer=cropped.write_to_memory(),
    dtype=np.uint8,
    shape=[cropped.height, cropped.width, cropped.bands],
)

# If RGB (3 bands), convert from RGB → BGR for OpenCV
if cropped.bands == 3:
    print("Chose cv")
    img = cv2.cvtColor(numpy_array, cv2.COLOR_RGB2BGR)
    del numpy_array
else:
    img = numpy_array  # e.g., grayscale or multispectral

del tissue_positions
gc.collect()

New image coordinates:  0 24437 12137 36829
Chose cv


12073

0 24437 12137 36829


### 4. Integrate gene expression and histology into a Graph

In [5]:
import os 

#Get coordinates
x_array=adata.obs["array_col"].tolist()
y_array=adata.obs["array_row"].tolist()
x_pixel_total=adata.obs["x_pixel"].tolist()
y_pixel_total=adata.obs["y_pixel"].tolist()

# Update coordinates based on relative pxl positions
x_pixel = [round(x - xmin) for x in x_pixel_total]
y_pixel = [round(y - ymin) for y in y_pixel_total]

#Test coordinates on the image
# Note: this results in 40x40 pixel black squares being drawn around each nucleus centroid
img_new=img.copy()
for i in range(len(x_pixel)):
    x=x_pixel[i]
    y=y_pixel[i]
    #img_new[int(x-20):int(x+20), int(y-20):int(y+20),:]=0
    img_new[int(y-20):int(y+20), int(x-20):int(x+20),:]=0


cv2.imwrite(os.path.join(results_path, f"sample_image_{person}.jpg"), img_new)
print(f"Wrote sample image to ", os.path.join(results_path, f"sample_image_{person}.jpg"))

del img_new
gc.collect()

Wrote sample image to  ../results/CRC_P1/sample_image_CRC_P1.jpg


0

- The ‘s’ parameter determines the weight given to histology when calculating Euclidean distance between every two spots. ‘s = 1’ means that the histology pixel intensity value has the same scale variance as the (x,y) coordinates, whereas higher value of ‘s’ indicates higher scale variance, hence, higher weight to histology, when calculating the Euclidean distance. 

- The "b"parameter determines the area of each spot when extracting color intensity.

In [18]:
#Calculate adjacent matrix
s=1
b=49
adj=spg.calculate_adj_matrix(x=x_pixel,y=y_pixel, x_pixel=x_pixel, y_pixel=y_pixel, image=img, beta=b, alpha=s, histology=True)
#If histlogy image is not available, SpaGCN can calculate the adjacent matrix using the fnction below
#adj=calculate_adj_matrix(x=x_pixel,y=y_pixel, histology=False)
np.savetxt(f'../data/{person}/adj.csv', adj, delimiter=',')

Calculateing adj matrix using histology image...
Var of c0,c1,c2 =  nan nan nan
Var of x,y,z =  35423906.91472115 26683469.673694864 nan


### 5. Spatial domain detection using SpaGCN

#### 5.1 Expression data preprocessing

In [4]:
adata=sc.read(os.path.join(local_data_path, "nuclei_centroids.h5ad"))
adj=np.loadtxt(os.path.join(local_data_path, 'adj.csv'), delimiter=',')
adata.var_names_make_unique()
spg.prefilter_genes(adata,min_cells=3) # avoiding all genes are zeros
spg.prefilter_specialgenes(adata)
#Normalize and take log for UMI
sc.pp.normalize_per_cell(adata)
sc.pp.log1p(adata)

#### 5.2 Set hyper-parameters

- p: Percentage of total expression contributed by neighborhoods.
- l: Parameter to control p.

In [8]:
p=0.5 
#Find the l value given p
l=spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)

Run 1: l [0.01, 1000], p [0.0, 153.8820492650696]
Run 2: l [0.01, 500.005], p [0.0, 28.015447343094223]
Run 3: l [0.01, 250.0075], p [0.0, 4.240330523308446]
Run 4: l [0.01, 125.00874999999999], p [0.0, 0.5157276735032843]
Run 5: l [62.509375, 125.00874999999999], p [0.028496868560644373, 0.5157276735032843]
Run 6: l [93.7590625, 125.00874999999999], p [0.18753135107474428, 0.5157276735032843]
Run 7: l [109.38390625, 125.00874999999999], p [0.32801349789332424, 0.5157276735032843]
Run 8: l [117.196328125, 125.00874999999999], p [0.4156469508032292, 0.5157276735032843]
Run 9: l [121.1025390625, 125.00874999999999], p [0.4640926787304587, 0.5157276735032843]
Run 10: l [123.05564453125, 125.00874999999999], p [0.48950676051756026, 0.5157276735032843]
recommended l =  124.032197265625


- n_clusters: Number of spatial domains wanted.
- res: Resolution in the initial Louvain's Clustering methods. If the number of clusters is known, we can use the spg.search_res() fnction to search for suitable resolution(optional)

In [9]:
#If the number of clusters known, we can use the spg.search_res() fnction to search for suitable resolution(optional)
#For this toy data, we set the number of clusters=7 since this tissue has 7 layers
n_clusters=7
#Set seed
r_seed=t_seed=n_seed=100
#Seaech for suitable resolution
res=spg.search_res(adata, adj, l, n_clusters, start=0.7, step=0.1, tol=5e-3, lr=0.05, max_epochs=20, r_seed=r_seed, t_seed=t_seed, n_seed=n_seed)

Start at res =  0.7 step =  0.1
Initializing cluster centers with louvain, resolution =  0.7
Epoch  0
Epoch  10
Res =  0.7 Num of clusters =  7
recommended res =  0.7


#### 5.3 Run SpaGCN

In [ ]:
clf=spg.SpaGCN()
clf.set_l(l)
#Set seed
random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)
#Run
clf.train(adata,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
y_pred, prob=clf.predict()
adata.obs["pred"]= y_pred
adata.obs["pred"]=adata.obs["pred"].astype('category')
#Do cluster refinement(optional)
#shape="hexagon" for Visium data, "square" for ST data.
adj_2d=spg.calculate_adj_matrix(x=x_array,y=y_array, histology=False)
refined_pred=spg.refine(sample_id=adata.obs.index.tolist(), pred=adata.obs["pred"].tolist(), dis=adj_2d, shape="hexagon")
adata.obs["refined_pred"]=refined_pred
adata.obs["refined_pred"]=adata.obs["refined_pred"].astype('category')
#Save results
adata.write_h5ad(os.path.join(results_path, "results.h5ad"))

Initializing cluster centers with louvain, resolution =  0.7
Epoch  0
Epoch  10
Epoch  20
Epoch  30
Epoch  40
Epoch  50
Epoch  60
Epoch  70
Epoch  80
Epoch  90
Epoch  100
Epoch  110
Epoch  120
Epoch  130
Epoch  140
Epoch  150
Epoch  160
Epoch  170
Epoch  180
Epoch  190
Calculateing adj matrix using xy only...


#### 5.4 Plot spatial domains

In [ ]:
adata=sc.read(os.path.join(results_path, "results.h5ad"))
#Set colors used
plot_color=["#F56867","#FEB915","#C798EE","#59BE86","#7495D3","#D1D1D1","#6D1A9C","#15821E","#3A84E6","#997273","#787878","#DB4C6C","#9E7A7A","#554236","#AF5F3C","#93796C","#F9BD3F","#DAB370","#877F6C","#268785"]
#Plot spatial domains
domains="pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="y_pixel",y="x_pixel",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig(os.path.join(results_path, "pred.png"), dpi=600)
plt.close()

#Plot refined spatial domains
domains="refined_pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="y_pixel",y="x_pixel",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig(os.path.join(results_path, "refined_pred.png"), dpi=600)
plt.close()

**Spatial Domains**![](./sample_results/pred.png) **Refined Spatial Domains**![](./sample_results/refined_pred.png)

### 6. Identify SVGs

In [12]:
#Read in raw data
raw=sc.read("../tutorial/data/151673/sample_data.h5ad")
raw.var_names_make_unique()
raw.obs["pred"]=adata.obs["pred"].astype('category')
raw.obs["x_array"]=raw.obs["x2"]
raw.obs["y_array"]=raw.obs["x3"]
raw.obs["x_pixel"]=raw.obs["x4"]
raw.obs["y_pixel"]=raw.obs["x5"]
#Convert sparse matrix to non-sparse
raw.X=(raw.X.A if issparse(raw.X) else raw.X)
raw.raw=raw
sc.pp.log1p(raw)

Variable names are not unique. To make them unique, call `.var_names_make_unique`.


- target: Target domain to identify SVGs.
- min_in_group_fraction: Minium in-group expression fraction.
- min_in_out_group_ratio: Miniumn (in-group expression fraction) / (out-group expression fraction).
- min_fold_change: Miniumn (in-group expression) / (out-group expression).
- r: Radius to detect a spot's neighboring spots.

In [15]:
#Use domain 0 as an example
target=0
#Set filtering criterials
min_in_group_fraction=0.8
min_in_out_group_ratio=1
min_fold_change=1.5
#Search radius such that each spot in the target domain has approximately 10 neighbors on average
adj_2d=spg.calculate_adj_matrix(x=x_array, y=y_array, histology=False)
start, end= np.quantile(adj_2d[adj_2d!=0],q=0.001), np.quantile(adj_2d[adj_2d!=0],q=0.1)
r=spg.search_radius(target_cluster=target, cell_id=adata.obs.index.tolist(), x=x_array, y=y_array, pred=adata.obs["pred"].tolist(), start=start, end=end, num_min=10, num_max=14,  max_run=100)
#Detect neighboring domains
nbr_domians=spg.find_neighbor_clusters(target_cluster=target,
                                   cell_id=raw.obs.index.tolist(), 
                                   x=raw.obs["x_array"].tolist(), 
                                   y=raw.obs["y_array"].tolist(), 
                                   pred=raw.obs["pred"].tolist(),
                                   radius=r,
                                   ratio=1/2)

nbr_domians=nbr_domians[0:3]
de_genes_info=spg.rank_genes_groups(input_adata=raw,
                                target_cluster=target,
                                nbr_list=nbr_domians, 
                                label_col="pred", 
                                adj_nbr=True, 
                                log=True)
#Filter genes
de_genes_info=de_genes_info[(de_genes_info["pvals_adj"]<0.05)]
filtered_info=de_genes_info
filtered_info=filtered_info[(filtered_info["pvals_adj"]<0.05) &
                            (filtered_info["in_out_group_ratio"]>min_in_out_group_ratio) &
                            (filtered_info["in_group_fraction"]>min_in_group_fraction) &
                            (filtered_info["fold_change"]>min_fold_change)]
filtered_info=filtered_info.sort_values(by="in_group_fraction", ascending=False)
filtered_info["target_dmain"]=target
filtered_info["neighbors"]=str(nbr_domians)
print("SVGs for domain ", str(target),":", filtered_info["genes"].tolist())

Calculateing adj matrix using xy only...
Calculateing adj matrix using xy only...
Calculateing adj matrix using xy only...
Run 1: radius [1.4142135381698608, 16.970561981201172], num_nbr [1.0, 321.149863760218]
Calculateing adj matrix using xy only...
Run 2: radius [1.4142135381698608, 9.192387759685516], num_nbr [1.0, 117.8283378746594]
Calculateing adj matrix using xy only...
Run 3: radius [1.4142135381698608, 5.303300648927689], num_nbr [1.0, 41.85013623978202]
Calculateing adj matrix using xy only...
Run 4: radius [1.4142135381698608, 3.3587570935487747], num_nbr [1.0, 20.04632152588556]
Calculateing adj matrix using xy only...
Run 5: radius [2.386485315859318, 3.3587570935487747], num_nbr [8.7574931880109, 20.04632152588556]
Calculateing adj matrix using xy only...
recommended radius =  2.8726212047040462 num_nbr=12.524523160762943
radius= 2.8726212047040462 average number of neighbors for each spot is 12.524523160762943
 Cluster 0 has neighbors:
Dmain  3 :  863
Dmain  2 :  517
SV

In [82]:
filtered_info

,genes,in_group_fraction,out_group_fraction,in_out_group_ratio,in_group_mean_exp,out_group_mean_exp,fold_change,pvals_adj,target_dmain,neighbors
0,CAMK2N1,1.000000,0.944964,1.058242,2.333675,1.578288,2.128434,1.656040e-11,0,"[3, 2]"
2,ENC1,0.998638,0.941848,1.060295,2.457791,1.696083,2.141931,1.552131e-03,0,"[3, 2]"
4,GPM6A,0.997275,0.922118,1.081505,2.224006,1.561187,1.940255,8.602227e-03,0,"[3, 2]"
6,ARPP19,0.982289,0.853583,1.150784,1.889256,1.272106,1.853637,4.823349e-02,0,"[3, 2]"
1,HPCAL1,0.851499,0.465213,1.830342,1.141321,0.406338,2.085448,9.706465e-05,0,"[3, 2]"


In [103]:
#Plot refinedspatial domains
color_self = clr.LinearSegmentedColormap.from_list('pink_green', ['#3AB370',"#EAE7CC","#FD1593"], N=256)
for g in filtered_info["genes"].tolist():
    raw.obs["exp"]=raw.X[:,raw.var.index==g]
    ax=sc.pl.scatter(raw,alpha=1,x="y_pixel",y="x_pixel",color="exp",title=g,color_map=color_self,show=False,size=100000/raw.shape[0])
    ax.set_aspect('equal', 'box')
    ax.axes.invert_yaxis()
    plt.savefig("./sample_results/"+g+".png", dpi=600)
    plt.close()


**CAMK2N1**![](./sample_results/CAMK2N1.png) **ENC1**![](./sample_results/ENC1.png) **GPM6A**![](./sample_results/GPM6A.png) **ARPP19**![](./sample_results/ARPP19.png) **HPCAL1**![](./sample_results/HPCAL1.png)

### 7. Identify Meta Gene

In [132]:
#Use domain 2 as an example
target=2
meta_name, meta_exp=spg.find_meta_gene(input_adata=raw,
                    pred=raw.obs["pred"].tolist(),
                    target_domain=target,
                    start_gene="GFAP",
                    mean_diff=0,
                    early_stop=True,
                    max_iter=3,
                    use_raw=False)

raw.obs["meta"]=meta_exp

Trying to set attribute `.obs` of view, copying.
Trying to set attribute `.obs` of view, copying.


Add gene:  MGP
Minus gene:  FTH1
Absolute mean change: 0.8913243
Number of non-target spots reduced to: 1888
Meta gene is:  GFAP+MGP-FTH1


Trying to set attribute `.obs` of view, copying.


Add gene:  MYL9
Minus gene:  MBP
Absolute mean change: 2.175557
Number of non-target spots reduced to: 563
Meta gene is:  GFAP+MGP-FTH1+MYL9-MBP
Add gene:  KRT8
Minus gene:  MT-ATP6
Absolute mean change: 2.8935516
Number of non-target spots reduced to: 111
Meta gene is:  GFAP+MGP-FTH1+MYL9-MBP+KRT8-MT-ATP6


In [133]:
#Plot meta gene
g="GFAP"
raw.obs["exp"]=raw.X[:,raw.var.index==g]
ax=sc.pl.scatter(raw,alpha=1,x="y_pixel",y="x_pixel",color="exp",title=g,color_map=color_self,show=False,size=100000/raw.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./sample_results/"+g+".png", dpi=600)
plt.close()

raw.obs["exp"]=raw.obs["meta"]
ax=sc.pl.scatter(raw,alpha=1,x="y_pixel",y="x_pixel",color="exp",title=meta_name,color_map=color_self,show=False,size=100000/raw.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./sample_results/meta_gene.png", dpi=600)
plt.close()


**start**![](./sample_results/GFAP.png) **meta gene**![](./sample_results/meta_gene.png)

### 8. Multiple tissue sections analysis
In this section, we show an example on how to analysis multiple adjacent tissue sections using SpaGCN.

**Mouse brain anterior**![](./sample_results/MA1_lowres.png) **Mouse brain posterior**![](./sample_results/MP1_lowres.png)

#### 8.1 Read in data

In [30]:
adata1=sc.read("./data/Mouse_brain/MA1.h5ad")
adata2=sc.read("./data/Mouse_brain/MP1.h5ad")
img1=cv2.imread("./data/Mouse_brain/MA1_histology.tif")
img2=cv2.imread("./data/Mouse_brain/MP1_histology.tif")

Variable names are not unique. To make them unique, call `.var_names_make_unique`.
Variable names are not unique. To make them unique, call `.var_names_make_unique`.


#### 8.2 Extract color intensity

In [31]:
b=49
s=1
x_pixel1=adata1.obs["x4"].tolist()
y_pixel1=adata1.obs["x5"].tolist()
adata1.obs["color"]=spg.extract_color(x_pixel=x_pixel1, y_pixel=y_pixel1, image=img1, beta=b)
z_scale=np.max([np.std(x_pixel1), np.std(y_pixel1)])*s
adata1.obs["z"]=(adata1.obs["color"]-np.mean(adata1.obs["color"]))/np.std(adata1.obs["color"])*z_scale

x_pixel2=adata2.obs["x4"].tolist()
y_pixel2=adata2.obs["x5"].tolist()
adata2.obs["color"]=spg.extract_color(x_pixel=x_pixel2, y_pixel=y_pixel2, image=img2, beta=b)
z_scale=np.max([np.std(x_pixel2), np.std(y_pixel2)])*s
adata2.obs["z"]=(adata2.obs["color"]-np.mean(adata2.obs["color"]))/np.std(adata2.obs["color"])*z_scale
del img1, img2

#### 8.3 Modify coordinates to combine 2 sections

In [32]:
from anndata import AnnData
adata1.obs["x_pixel"]=x_pixel1
adata1.obs["y_pixel"]=y_pixel1
adata2.obs["x_pixel"]=x_pixel2-np.min(x_pixel2)+np.min(x_pixel1)
adata2.obs["y_pixel"]=y_pixel2-np.min(y_pixel2)+np.max(y_pixel1)
adata1.var_names_make_unique()
adata2.var_names_make_unique()
adata_all=AnnData.concatenate(adata1, adata2,join='inner',batch_key="dataset_batch",batch_categories=["0","1"])

#### 8.4 Integrate gene expression and histology into a Graph

In [34]:
X=np.array([adata_all.obs["x_pixel"], adata_all.obs["y_pixel"], adata_all.obs["z"]]).T.astype(np.float32)
adj=spg.pairwise_distance(X)

#### 8.5 Spatial domain detection using SpaGCN

In [37]:
sc.pp.normalize_per_cell(adata_all, min_counts=0)
sc.pp.log1p(adata_all)
p=0.5 
#Find the l value given p
l=spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)

Run 1: l [0.01, 1000], p [0.0, 144.17116893743565]
Run 2: l [0.01, 500.005], p [0.0, 24.78992462158203]
Run 3: l [0.01, 250.0075], p [0.0, 3.649960994720459]
Run 4: l [125.00874999999999, 250.0075], p [0.4487175941467285, 3.649960994720459]
Run 5: l [125.00874999999999, 187.508125], p [0.4487175941467285, 1.5741894245147705]
Run 6: l [125.00874999999999, 156.2584375], p [0.4487175941467285, 0.9070142507553101]
Run 7: l [125.00874999999999, 140.63359375], p [0.4487175941467285, 0.6537595987319946]
Run 8: l [125.00874999999999, 132.821171875], p [0.4487175941467285, 0.5454769134521484]
recommended l =  128.91496093749998


In [43]:
res=1.0
seed=100
random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
clf=spg.SpaGCN()
clf.set_l(l)
clf.train(adata_all,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
y_pred, prob=clf.predict()
adata_all.obs["pred"]= y_pred
adata_all.obs["pred"]=adata_all.obs["pred"].astype('category')

Initializing cluster centers with louvain, resolution =  1.0
Epoch  0
Epoch  10
Epoch  20
Epoch  30
delta_label  0.004794180856339891 < tol  0.005
Reach tolerance threshold. Stopping training.
Total epoch: 37


In [44]:
colors_use=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#bec1d4', '#bb7784', '#0000ff', '#111010', '#FFFF00',   '#1f77b4', '#800080', '#959595', 
 '#7d87b9', '#bec1d4', '#d6bcc0', '#bb7784', '#8e063b', '#4a6fe3', '#8595e1', '#b5bbe3', '#e6afb9', '#e07b91', '#d33f6a', '#11c638', '#8dd593', '#c6dec7', '#ead3c6', '#f0b98d', '#ef9708', '#0fcfc0', '#9cded6', '#d5eae7', '#f3e1eb', '#f6c4e1', '#f79cd4']
num_celltype=len(adata_all.obs["pred"].unique())
adata_all.uns["pred_colors"]=list(colors_use[:num_celltype])
ax=sc.pl.scatter(adata_all,alpha=1,x="y_pixel",y="x_pixel",color="pred",show=False,size=150000/adata_all.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./sample_results/mouse_barin_muti_sections_domains.png", dpi=600)
plt.close()

**SpaGCN mouse brain combines**![](./sample_results/mouse_barin_muti_sections_domains.png)